# Evaluating Scorio GPQA with `scorio.eval`

This notebook evaluates ten questions from one field. The response matrix has shape
`10 x 80`: questions by attempts. Change
`field_start` to any multiple of 50 from 0 through 3550 to sample another field.

Only correctness and identity columns are read.


In [1]:
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

from scorio import eval

BUCKET_ROOT = "hf://buckets/harimo/scorio-gpqa"


def pool_path(model, question_id):
    return f"{BUCKET_ROOT}/data/{model}/super_gpqa/q{question_id:04d}.parquet"


def read_pools(model, question_ids, columns, max_workers=2):
    """Read selected columns from question files, preserving question order."""
    paths = [pool_path(model, q) for q in question_ids]

    def read_one(path):
        return pq.read_table(path, columns=columns)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        tables = list(executor.map(read_one, paths))
    return pa.concat_tables(tables).to_pandas()

models = ["Qwen3.6-35B-A3B", "gpt-oss-20b_low", "gpt-oss-20b_medium", "gpt-oss-20b_high"]
model_name = "gpt-oss-20b_medium"
field_start = 0
question_count = 10
question_ids = range(field_start, field_start + question_count)
columns = ["full_data_id", "seed", "field", "evalscope_is_correct"]

rows = read_pools(model_name, question_ids, columns).sort_values(["full_data_id", "seed"])
assert rows.groupby("full_data_id").size().eq(80).all()
R = rows.evalscope_is_correct.to_numpy().astype(int).reshape(question_count, 80)

print("field:", rows.field.iloc[0])
print(R.shape, R.dtype)


/tmp/scorio_uv_cache/archive-v0/qlDst6OSBXQ1PndVk4Wef/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


field: Aeronautical and Astronautical Science and Technology
(10, 80) int64


## Bayes@N

Bayes@N estimates expected accuracy from `N` attempts per question. It applies a uniform
Beta(1, 1) prior to each question's success rate, avoiding estimates of exactly zero or one
from finite samples. `bayes_ci` returns the posterior mean, posterior standard deviation, and
a normal-approximation 95% credible interval across the questions.


In [2]:
mu, sigma, lo, hi = eval.bayes_ci(R)
print(f"Bayes@N: {mu:.3f} +- {sigma:.3f}")
print(f"95% credible interval: [{lo:.3f}, {hi:.3f}]")


Bayes@N: 0.374 +- 0.011
95% credible interval: [0.353, 0.396]


## Pass@k, Maj@k, and Pass^k


In [3]:
ks = [1, 2, 4, 8, 16, 80]
table = pd.DataFrame({
    "pass@k": [eval.pass_at_k(R, k) for k in ks],
    "maj@k": [eval.maj_at_k(R, k) for k in ks],
    "pass^k": [eval.pass_hat_k(R, k) for k in ks],
    "auc@k": [eval.auc_at_k(R, k) for k in ks],
}, index=pd.Index(ks, name="k"))
display(table.round(3))


,pass@k,maj@k,pass^k,auc@k
k,,,,
1,0.371,0.371,0.371,0.371
2,0.464,0.279,0.279,0.417
4,0.546,0.318,0.208,0.478
8,0.615,0.330,0.142,0.539
16,0.673,0.326,0.080,0.597
80,0.900,0.300,0.000,0.760


## Sample-budget sweep


In [4]:
budgets = [1, 2, 4, 8, 16, 32, 80]
sweep = pd.DataFrame(
    [eval.bayes_ci(R[:, :n]) for n in budgets],
    columns=["mu", "sigma", "lo", "hi"],
    index=pd.Index(budgets, name="samples"),
)
sweep["width"] = sweep.hi - sweep.lo
display(sweep.round(3))


,mu,sigma,lo,hi,width
samples,,,,,
1,0.433,0.075,0.287,0.579,0.292
2,0.425,0.062,0.303,0.547,0.244
4,0.433,0.050,0.336,0.531,0.194
8,0.390,0.037,0.318,0.462,0.144
16,0.367,0.025,0.318,0.415,0.097
32,0.362,0.018,0.327,0.396,0.069
80,0.374,0.011,0.353,0.396,0.043


## Compare all four configurations on the same field


In [5]:
matrices = []
for model in models:
    if model == model_name:
        matrices.append(R)
        continue
    one = read_pools(model, question_ids, columns).sort_values(["full_data_id", "seed"])
    matrices.append(one.evalscope_is_correct.to_numpy().astype(int).reshape(question_count, 80))

comparison = pd.DataFrame(
    [eval.bayes_ci(matrix) for matrix in matrices],
    columns=["mu", "sigma", "lo", "hi"],
    index=models,
)
display(comparison.sort_values("mu", ascending=False).round(3))


HTTP Error 429 thrown while requesting GET https://huggingface.co/buckets/harimo/scorio-gpqa/resolve/data/Qwen3.6-35B-A3B/super_gpqa/q0001.parquet


Retrying in 1s [Retry 1/5].


HTTP Error 429 thrown while requesting GET https://huggingface.co/buckets/harimo/scorio-gpqa/resolve/data/Qwen3.6-35B-A3B/super_gpqa/q0001.parquet


Retrying in 2s [Retry 2/5].


HTTP Error 429 thrown while requesting GET https://huggingface.co/buckets/harimo/scorio-gpqa/resolve/data/Qwen3.6-35B-A3B/super_gpqa/q0001.parquet


...

'The read operation timed out' thrown while requesting GET https://huggingface.co/buckets/harimo/scorio-gpqa/resolve/data/gpt-oss-20b_high/super_gpqa/q0003.parquet


Retrying in 2s [Retry 2/5].


'The read operation timed out' thrown while requesting GET https://huggingface.co/buckets/harimo/scorio-gpqa/resolve/data/gpt-oss-20b_high/super_gpqa/q0002.parquet


Retrying in 2s [Retry 2/5].


,mu,sigma,lo,hi
Qwen3.6-35B-A3B,0.578,0.011,0.556,0.600
gpt-oss-20b_high,0.437,0.009,0.418,0.455
gpt-oss-20b_medium,0.374,0.011,0.353,0.396
gpt-oss-20b_low,0.233,0.013,0.208,0.258


The [evaluation reference](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/README.md)
lists the remaining estimators and their sources.
